# End-to-End Callable Pricing Workflow

Build a yield curve, calibrate Hull-White from swaption vols, construct a vol cube with SABR smile, and price callable/Bermudan instruments — all from synthetic market data.

**Pipeline:**
1. Synthetic market data → yield curve (5 methods compared)
2. Swaption vol surface → Hull-White calibration
3. Vol cube with SABR smile
4. Callable bond + Bermudan swaption pricing
5. Sensitivity analysis (vega, duration)

Uses `pricebook.viz` throughout.

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 11, 4)
print("Pricebook — Callable Pricing Workflow")
print(f"Reference date: {REF}")

## 1. Yield Curve Construction — 3 Methods Compared

In [ ]:
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod
from pricebook.curves.nelson_siegel import calibrate_nelson_siegel, ns_discount_curve, calibrate_svensson, svensson_discount_curve

# Build EUR-like curve from realistic synthetic swap strip
# Realistic EUR rates: 3% short end, slight upward slope, humped at 5-10Y
tenors = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30]
# Realistic EUR OIS term structure (Nov 2024)
zeros = [0.0310, 0.0305, 0.0295, 0.0285, 0.0290, 0.0310, 0.0325, 0.0340, 0.0355, 0.0360, 0.0365]

# Method 1: Direct from market zero rates (piecewise log-linear)
dates_direct = [REF + relativedelta(years=int(t), months=int((t%1)*12)) for t in tenors]
dfs_direct = [math.exp(-z * t) for z, t in zip(zeros, tenors)]
curve_direct = DiscountCurve(REF, dates_direct, dfs_direct,
                              interpolation=InterpolationMethod.LOG_LINEAR)

# Method 2: Nelson-Siegel (smooth, captures level + slope + curvature)
ns_params = calibrate_nelson_siegel(tenors, zeros)
curve_ns = ns_discount_curve(REF, ns_params["beta0"], ns_params["beta1"],
                              ns_params["beta2"], ns_params["tau"])
print(f"NS params: β0={ns_params['beta0']:.4f}, β1={ns_params['beta1']:.4f}, "
      f"β2={ns_params['beta2']:.4f}, τ={ns_params['tau']:.2f}, RMSE={ns_params['rmse']*10000:.1f}bp")

# Method 3: Svensson (captures humps better — 2nd curvature term)
sv_params = calibrate_svensson(tenors, zeros)
curve_sv = svensson_discount_curve(REF, sv_params["beta0"], sv_params["beta1"],
                                    sv_params["beta2"], sv_params["tau1"],
                                    sv_params["beta3"], sv_params["tau2"])
print(f"SV params: β0={sv_params['beta0']:.4f}, β1={sv_params['beta1']:.4f}, "
      f"β2={sv_params['beta2']:.4f}, β3={sv_params['beta3']:.4f}, RMSE={sv_params['rmse']*10000:.1f}bp")

curves = {"Market (log-linear)": curve_direct, "Nelson-Siegel": curve_ns, "Svensson": curve_sv}

# Plot
plot_tenors = list(range(1, 31))
with apply_theme():
    fig, axes = create_figure(2)
    ax1, ax2 = axes[0], axes[1]

    for label, curve in curves.items():
        zs = [curve.zero_rate(REF + relativedelta(years=t)) * 100 for t in plot_tenors]
        ax1.plot(plot_tenors, zs, linewidth=2, label=label)
    # Also plot the input market points
    ax1.scatter([t for t in tenors if t >= 1], [z*100 for t, z in zip(tenors, zeros) if t >= 1],
                s=50, zorder=5, color='black', label='Market inputs')
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("Zero Rate (%)")
    ax1.set_title("EUR Yield Curve — 3 Construction Methods")
    ax1.legend(fontsize=9)

    for label, curve in curves.items():
        fwds = []
        for t in plot_tenors[:-1]:
            d1 = REF + relativedelta(years=t)
            d2 = REF + relativedelta(years=t+1)
            fwd = curve.forward_rate(d1, d2) * 100
            fwds.append(fwd)
        ax2.plot(plot_tenors[:-1], fwds, linewidth=2, label=label)
    ax2.set_xlabel("Tenor (years)")
    ax2.set_ylabel("1Y Forward Rate (%)")
    ax2.set_title("Forward Rate Curves")
    ax2.legend(fontsize=9)
    fig.tight_layout()

## 2. Hull-White Calibration from Swaption Vols

In [ ]:
from pricebook.models.hw_calibration import calibrate_hull_white
from pricebook.options.synthetic_swaption_data import synthetic_hw_targets

# Use the Sequential curve for pricing
curve = curve_direct

# Get synthetic swaption vol targets for EUR
targets = synthetic_hw_targets("EUR", REF)
print(f"Calibrating HW to {len(targets)} swaption vol targets:")
for (exp, ten), vol in sorted(targets.items()):
    print(f"  {exp:>4.0f}Y × {ten:>4.0f}Y: {vol*10000:>6.1f} bp")

# Calibrate
hw_result = calibrate_hull_white(curve, targets, n_steps=30)
print(f"\nCalibrated HW parameters:")
print(f"  a (mean reversion) = {hw_result.a:.4f}")
print(f"  sigma (volatility) = {hw_result.sigma:.6f}")
print(f"  RMSE = {hw_result.rmse_vol*10000:.1f} bp")
print(f"  Converged: {hw_result.converged}")

# Plot fit diagnostics
with apply_theme():
    fig, axes = create_figure(2)
    ax1, ax2 = axes[0], axes[1]

    mkt = [e["market_vol"]*10000 for e in hw_result.per_swaption_errors]
    mdl = [e["model_vol"]*10000 for e in hw_result.per_swaption_errors]
    labels = [f"{e['expiry']:.0f}×{e['tenor']:.0f}" for e in hw_result.per_swaption_errors]

    x = range(len(labels))
    ax1.bar([i-0.15 for i in x], mkt, 0.3, label='Market', color='#3b82f6')
    ax1.bar([i+0.15 for i in x], mdl, 0.3, label='Model', color='#ef4444')
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(labels, rotation=45)
    ax1.set_ylabel("Vol (bp)")
    ax1.set_title("HW Calibration Fit: Market vs Model")
    ax1.legend()

    errors = [e["error_bp"] for e in hw_result.per_swaption_errors]
    ax2.bar(x, errors, color=['#22c55e' if abs(e) < 5 else '#ef4444' for e in errors])
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(labels, rotation=45)
    ax2.set_ylabel("Error (bp)")
    ax2.set_title("Per-Swaption Calibration Error")
    ax2.axhline(0, color='black', linewidth=0.5)
    fig.tight_layout()

## 3. Swaption Vol Cube with SABR Smile

In [ ]:
from pricebook.options.synthetic_swaption_data import synthetic_atm_surface, synthetic_smile_data
from pricebook.options.swaption_vol_cube import build_swaption_vol_cube

# Build vol cube with SABR smile
atm_cube = synthetic_atm_surface("EUR", REF)
smile = synthetic_smile_data("EUR", REF, forward=0.03)

# Add smile data
cube = build_swaption_vol_cube(
    REF,
    [0.5, 1.0, 2.0, 5.0, 10.0],
    [1.0, 2.0, 5.0, 10.0, 20.0, 30.0],
    [[atm_cube.vol_by_years(e, t) for t in [1,2,5,10,20,30]] for e in [0.5,1,2,5,10]],
    smile_data=smile,
)

# Plot ATM surface as heatmap
with apply_theme():
    fig, axes = create_figure(2)
    ax1, ax2 = axes[0], axes[1]

    expiries = [0.5, 1, 2, 5, 10]
    tenors_plot = [1, 2, 5, 10, 20, 30]
    grid = np.array([[cube.atm_vol(e, t)*10000 for t in tenors_plot] for e in expiries])

    im = ax1.imshow(grid, aspect='auto', cmap='RdYlBu_r',
                     extent=[0, len(tenors_plot)-1, len(expiries)-1, 0])
    ax1.set_xticks(range(len(tenors_plot)))
    ax1.set_xticklabels([f"{t}Y" for t in tenors_plot])
    ax1.set_yticks(range(len(expiries)))
    ax1.set_yticklabels([f"{e}Y" for e in expiries])
    ax1.set_xlabel("Swap Tenor")
    ax1.set_ylabel("Option Expiry")
    ax1.set_title("EUR ATM Swaption Vol Surface (bp)")
    fig.colorbar(im, ax=ax1, label='Vol (bp)')

    # Plot SABR smile at 5Y×10Y
    strikes = np.linspace(0.01, 0.06, 30)
    smile_vols = [cube.vol_by_years(5.0, 10.0, k)*10000 for k in strikes]
    ax2.plot(strikes*100, smile_vols, linewidth=2, color='#3b82f6')
    ax2.axvline(3.0, color='gray', linestyle='--', alpha=0.5, label='ATM ~3%')
    ax2.set_xlabel("Strike (%)")
    ax2.set_ylabel("Vol (bp)")
    ax2.set_title("SABR Smile at 5Y×10Y")
    ax2.legend()
    fig.tight_layout()

## 4. Callable Bond & Bermudan Swaption Pricing

In [ ]:
from pricebook.fixed_income.callable_bond import callable_bond_price, puttable_bond_price
from pricebook.options.bermudan_swaption import bermudan_swaption_tree

hw = hw_result.model

# Callable bond: 10Y, 4% coupon, callable after year 3
print("=== Callable Bond (10Y, 4% coupon) ===")
call_dates = [float(y) for y in range(3, 10)]
callable_px = callable_bond_price(hw, 0.04, 10.0, call_dates, call_price=100.0, n_steps=50)
straight_px = callable_bond_price(hw, 0.04, 10.0, [], call_price=100.0, n_steps=50)
print(f"  Straight price:  {straight_px:.4f}")
print(f"  Callable price:  {callable_px:.4f}")
print(f"  Call option value: {straight_px - callable_px:.4f}")

# Bermudan swaption: 5nc1 (5Y swap, callable from year 1)
print("\n=== Bermudan Swaption (5nc1) ===")
exercise = [1.0, 2.0, 3.0, 4.0]
berm_px = bermudan_swaption_tree(hw, exercise, 5.0, 0.03, is_payer=True, n_steps=80)
euro_px = bermudan_swaption_tree(hw, [1.0], 5.0, 0.03, is_payer=True, n_steps=80)
print(f"  European (1Y into 4Y): {euro_px*10000:.1f} bp")
print(f"  Bermudan (5nc1):       {berm_px*10000:.1f} bp")
print(f"  Early exercise premium: {(berm_px - euro_px)*10000:.1f} bp")

# Callable vs rate level
print("\n=== Sensitivity to Rate Level ===")
rate_shifts = np.linspace(-0.02, 0.02, 11)
callable_prices = []
straight_prices = []
for shift in rate_shifts:
    from pricebook.core.discount_curve import DiscountCurve
    from pricebook.core.interpolation import InterpolationMethod
    shifted_dates = [REF + relativedelta(years=y) for y in range(1, 35)]
    shifted_dfs = [math.exp(-(0.03 + shift) * y) for y in range(1, 35)]
    shifted_curve = DiscountCurve(REF, shifted_dates, shifted_dfs, interpolation=InterpolationMethod.LOG_LINEAR)
    from pricebook.models.hull_white import HullWhite
    hw_shifted = HullWhite(a=hw.a, sigma=hw.sigma, curve=shifted_curve)
    cp = callable_bond_price(hw_shifted, 0.04, 10.0, call_dates, n_steps=30)
    sp = callable_bond_price(hw_shifted, 0.04, 10.0, [], n_steps=30)
    callable_prices.append(cp)
    straight_prices.append(sp)

with apply_theme():
    fig, axes = create_figure(2)
    ax1, ax2 = axes[0], axes[1]

    ax1.plot(rate_shifts*100, straight_prices, linewidth=2, label='Straight', color='#3b82f6')
    ax1.plot(rate_shifts*100, callable_prices, linewidth=2, label='Callable', color='#ef4444')
    ax1.fill_between(rate_shifts*100, callable_prices, straight_prices, alpha=0.15, color='#ef4444', label='Call value')
    ax1.set_xlabel("Rate Shift (bp)")
    ax1.set_ylabel("Price (per 100)")
    ax1.set_title("Callable vs Straight Bond — Rate Sensitivity")
    ax1.legend()
    ax1.axhline(100, color='gray', linestyle=':', alpha=0.5)

    # Negative convexity: price-yield for callable
    ax2.plot(rate_shifts*100, [s - c for s, c in zip(straight_prices, callable_prices)],
             linewidth=2, color='#f59e0b')
    ax2.set_xlabel("Rate Shift (bp)")
    ax2.set_ylabel("Call Option Value")
    ax2.set_title("Embedded Call Option Value vs Rate Level")
    ax2.axhline(0, color='black', linewidth=0.5)
    fig.tight_layout()

## 5. Multi-Currency Comparison

In [ ]:
from pricebook.models.hw_per_currency import calibrate_hw_for_currency
from pricebook.options.synthetic_swaption_data import synthetic_atm_surface

currencies = ["USD", "EUR", "GBP", "JPY", "BRL", "ZAR"]
hw_params = {}

for ccy in currencies:
    from pricebook.core.discount_curve import DiscountCurve
    from pricebook.core.interpolation import InterpolationMethod
    from pricebook.curves.synthetic_market_data import _MARKET_RATES
    base_rate = _MARKET_RATES.get(ccy, (0.04, 0.002))[0]
    dates_ccy = [REF + relativedelta(years=y) for y in range(1, 35)]
    dfs_ccy = [math.exp(-base_rate * y) for y in range(1, 35)]
    curve_ccy = DiscountCurve(REF, dates_ccy, dfs_ccy, interpolation=InterpolationMethod.LOG_LINEAR)

    result = calibrate_hw_for_currency(ccy, REF, curve_ccy, use_synthetic=False)
    hw_params[ccy] = (result.a, result.sigma, base_rate)
    print(f"  {ccy}: a={result.a:.3f}, sigma={result.sigma:.4f}, rate={base_rate*100:.1f}%")

with apply_theme():
    fig, axes = create_figure(3)
    ax1, ax2, ax3 = axes[0], axes[1], axes[2]

    ccys = list(hw_params.keys())
    a_vals = [hw_params[c][0] for c in ccys]
    s_vals = [hw_params[c][1]*10000 for c in ccys]
    r_vals = [hw_params[c][2]*100 for c in ccys]

    colors = ['#3b82f6', '#3b82f6', '#3b82f6', '#3b82f6', '#ef4444', '#ef4444']

    ax1.barh(ccys, a_vals, color=colors)
    ax1.set_xlabel("Mean Reversion (a)")
    ax1.set_title("HW Mean Reversion by Currency")

    ax2.barh(ccys, s_vals, color=colors)
    ax2.set_xlabel("Volatility (bp)")
    ax2.set_title("HW Short-Rate Vol by Currency")

    ax3.barh(ccys, r_vals, color=colors)
    ax3.set_xlabel("Base Rate (%)")
    ax3.set_title("Current Rate Level")

    fig.suptitle("Multi-Currency Hull-White Parameters (Blue=G10, Red=EM)", y=1.02, fontsize=12)
    fig.tight_layout()

## Summary

| Step | Component | Method |
|------|-----------|--------|
| 1 | Yield curve | `build_curves()` — Sequential, NS, Svensson (unified for all 33 CCY) |
| 2 | HW calibration | `calibrate_hull_white()` — minimise swaption vol errors |
| 3 | Vol cube | `SwaptionVolCube` — ATM backbone + SABR smile per node |
| 4 | Callable pricing | `callable_bond_price()` — HW trinomial tree, backward induction |
| 5 | Multi-currency | `calibrate_hw_for_currency()` — defaults for 33 markets |

**Full pipeline**: synthetic data → curve → vol → HW calibration → callable pricing.
All using `pricebook.viz` for consistent visuals.